# Train — VinRobotics mjlab (Google Colab)

Notebook chuyển từ `scripts/train.py` sang dạng chạy trong Colab.
Thay vì parse tham số dòng lệnh (`tyro.cli`), ta dựng `TrainConfig` trực tiếp bằng Python rồi chỉnh field.

**Yêu cầu:** runtime có GPU (Runtime → Change runtime type → GPU).


## 1. Cài đặt & lấy source

In [ ]:
# # Nếu chạy trên Colab thuần: clone repo. Nếu đã kết nối runtime tới thư mục local (VS Code Colab extension), bỏ qua cell clone.
# import os

# REPO_URL = ""  # ví dụ: https://github.com/<org>/vinrobotics_mjlab.git
# REPO_DIR = "/content/vinrobotics_mjlab"

# if REPO_URL and not os.path.isdir(REPO_DIR):
#     os.system(f"git clone {REPO_URL} {REPO_DIR}")

# # Chuyển working dir về repo root (chỉnh lại nếu đường dẫn khác).
# for cand in (REPO_DIR, os.getcwd()):
#     if os.path.isfile(os.path.join(cand, "setup.py")):
#         os.chdir(cand)
#         break
# print("cwd:", os.getcwd())

In [ ]:
# Cài dependencies (bỏ comment khi chạy lần đầu trên Colab).
# !pip install -q mjlab==1.4.0 mujoco==3.8.1 mujoco-warp==3.8.1 warp-lang==1.13.0
# !pip install -q -e .

## 2. Thiết lập môi trường render (headless GPU)

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())

# Colab không có màn hình -> dùng EGL để render bằng GPU.
os.environ["MUJOCO_GL"] = "egl"
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
print("MUJOCO_GL =", os.environ["MUJOCO_GL"])

## 3. Chọn task & liệt kê task khả dụng

In [ ]:
import mjlab.tasks  # noqa: F401  (populate registry)
import src.tasks    # noqa: F401
from mjlab.tasks.registry import list_tasks

print("Available tasks:")
for t in list_tasks():
    print("  -", t)

TASK_ID = "VR-M3-1-12DOF-Flat"  # đổi task tại đây

## 4. Dựng cấu hình huấn luyện

`TrainConfig.from_task` tạo config mặc định; chỉnh các field bên dưới thay cho cờ dòng lệnh.


In [ ]:
from scripts.train import TrainConfig

cfg = TrainConfig.from_task(TASK_ID)

# --- Chỉnh tham số (tương đương các cờ CLI) ---
cfg.env.scene.num_envs = 4096          # --env.scene.num-envs
cfg.agent.max_iterations = 1000        # số vòng học
cfg = __import__("dataclasses").replace(cfg, gpu_ids=[0], video=False)

print("task            :", TASK_ID)
print("num_envs        :", cfg.env.scene.num_envs)
print("max_iterations  :", cfg.agent.max_iterations)
print("experiment_name :", cfg.agent.experiment_name)

> Lưu ý: `TrainConfig` là dataclass `frozen`. Với field của chính `TrainConfig` (vd `gpu_ids`, `video`)
> phải dùng `dataclasses.replace`. Các object con (`cfg.env`, `cfg.agent`) vẫn gán trực tiếp được.


## 5. Chạy huấn luyện

In [ ]:
from scripts.train import launch_training

launch_training(task_id=TASK_ID, args=cfg)

## 6. Kết quả

Checkpoint lưu tại `logs/rsl_rl/<experiment_name>/<date_time>/model_<iter>.pt`,
kèm `policy.onnx` để deploy. Dùng đường dẫn này trong `play.ipynb`.


In [ ]:
import glob
ckpts = sorted(glob.glob(f"logs/rsl_rl/{cfg.agent.experiment_name}/**/model_*.pt", recursive=True))
print("Checkpoints:")
for c in ckpts[-5:]:
    print("  ", c)